# Train a Machine Learning Model
Determine the optimal machine learning algorithm and train it.

In [2]:
%matplotlib inline
import sys
import os
sys.path.append(f"./Al_data")
from glob import glob
from copy import deepcopy
from glob import glob
from matminer.featurizers.function import FunctionFeaturizer
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from scipy import stats
from sklearn.dummy import DummyRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import cross_validate, cross_val_predict, RepeatedKFold, GridSearchCV
from sklearn.linear_model import BayesianRidge, LinearRegression, Lasso, LassoLars
from sklearn.decomposition import PCA
from sklearn.preprocessing import PolynomialFeatures, MinMaxScaler
from sklearn.pipeline import Pipeline
from stopping_power_ml.utils.io import load_qbox_data
import pandas as pd
import numpy as np
import pickle as pkl
import tensorflow as tf

from tensorflow import keras

from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

2025-07-14 14:26:01.168756: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-14 14:26:01.172356: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-14 14:26:01.181424: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752521161.198487 2434905 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752521161.203598 2434905 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1752521161.217831 2434905 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [12]:
sys.path.append('./stopping-power-ML-reu/')
from nn_utilities import plot_training_history, plot_loglog_mae, prepare_sequence_datasets

In [13]:
import warnings; warnings.simplefilter('ignore')

## Load in the Dataset
This was created by a different notebook

In [14]:
data = pd.read_pickle(os.path.join('data', '0_2_atom_ion_mag', 'channel_data_0_2_atom_ion_mag.pkl.gz'))
print('Data set size:', len(data))
data.head()

Data set size: 2000


,frame_id,force,position,velocity,energy,file_id,file,timestep,displacement,directory,...,AGNI_z in Al eta=6.80e+00,AGNI_x in Al eta=1.04e+01,AGNI_y in Al eta=1.04e+01,AGNI_z in Al eta=1.04e+01,AGNI_x in Al eta=1.60e+01,AGNI_y in Al eta=1.60e+01,AGNI_z in Al eta=1.60e+01,ion-ion repulsion,initial,average_range
50769,0,0.000139,"[0.0, 5.74196597, 5.74196597]",1.0,-18651.709649,1,datasets/Al_256_channel/Dv1.0,0,0.000000,datasets/Al_256_channel/Dv1.0,...,-0.000002,0.000010,-0.000006,-0.000006,0.000011,-0.000007,-0.000007,0.000018,True,False
50770,1,0.017550,"[0.014284, 5.74196597, 5.74196597]",1.0,-18651.702070,1,datasets/Al_256_channel/Dv1.0,1,0.014284,datasets/Al_256_channel/Dv1.0,...,-0.000002,0.001114,-0.000006,-0.000006,0.001191,-0.000007,-0.000007,0.007413,True,False
50771,2,0.034775,"[0.028568, 5.74196597, 5.74196597]",1.0,-18651.701695,1,datasets/Al_256_channel/Dv1.0,2,0.028568,datasets/Al_256_channel/Dv1.0,...,-0.000002,0.002217,-0.000006,-0.000006,0.002370,-0.000007,-0.000007,0.014826,True,False
50772,3,0.051492,"[0.042852, 5.74196597, 5.74196597]",1.0,-18651.701075,1,datasets/Al_256_channel/Dv1.0,3,0.042852,datasets/Al_256_channel/Dv1.0,...,-0.000002,0.003318,-0.000006,-0.000006,0.003546,-0.000007,-0.000007,0.022230,True,False
50773,4,0.067534,"[0.057136, 5.74196597, 5.74196597]",1.0,-18651.700217,1,datasets/Al_256_channel/Dv1.0,4,0.057136,datasets/Al_256_channel/Dv1.0,...,-0.000002,0.004418,-0.000006,-0.000006,0.004719,-0.000007,-0.000007,0.029620,True,False


Remove the initial transient

In [15]:
data.query('initial == False', inplace=True)
print('Training set size:', len(data))

Training set size: 1579


Determine which columns are inputs

In [16]:
featurizers = pkl.load(open(os.path.join('data', 'featurizers_chrg_1_atoms.pkl'), 'rb'))

In [17]:
X_cols = featurizers.feature_labels()

Determine which column is target

In [18]:
y_col = 'force'
print(y_col)

force


Following function prepares the input data (`X`) and target labels (`y`) for training a recurrent neural network (RNN). A **sliding window** is applied to extract sequences of input features.


In [19]:
train_ds, val_ds, x_scaler, y_scaler, input_shape = prepare_sequence_datasets(
    data=data,
    X_cols=X_cols,
    y_col='force',
    timesteps=10,
    batch_size=64
)

2025-07-14 14:29:05.044165: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## A quick look at auto correlation
To determine if we should use a time-series optimized model such as convolutional or recurrent neural neworks, we see if energy and force (the two possible targets to predict) are correlated with themselves over time. We hypothesize that they are.


In [20]:
force_autocorr = data['force'].autocorr(lag=1)
print(f'Autocorrelation of force (lag=1): {force_autocorr:.4f}')

Autocorrelation of force (lag=1): 0.9997


In [21]:
energy_autocorr = data['energy'].autocorr(lag=1)
print(f'Autocorrelation of energy (lag=1): {energy_autocorr:.4f}')

Autocorrelation of energy (lag=1): 1.0000


This indeed shows a very strong correlation for force and a perfect correlation for energy. We will pursue models which work best for time-series data.

## Tune Hyperparameters for Transformer Model

In [16]:
import os
import datetime
import random
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
from tensorflow.keras.layers import Layer, Embedding, Input, Dense, Dropout, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LayerNormalization, Dense, Dropout, MultiHeadAttention, Layer, Input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Embedding, Add, Layer
import tensorflow.keras.backend as K

# l1, recurrent, kernel reglulizers (lots of ranges), batch size, timestep

GOOD_EMOJIS = ["🔥", "🚀", "🌟", "🎉", "✨", "🤑", "👌"]
MEDIUM_EMOJIS = ["🤔", "🔍", "🕵️", "🧩", "🔎", "🛠️"]
BAD_EMOJIS = ["💀", "😵", "☠️", "😞", "😟", "⚠️"]

all_maes = []
def pick_emoji(mae, best_mae, worst_mae):
    # Normalize mae between best and worst (0 to 1)
    if worst_mae == best_mae:
        score = 0  # avoid division by zero if all equal
    else:
        score = (mae - best_mae) / (worst_mae - best_mae)

    if score <= 0.33:
        return random.choice(GOOD_EMOJIS)
    elif score <= 0.66:
        return random.choice(MEDIUM_EMOJIS)
    else:
        return random.choice(BAD_EMOJIS)


class TransformerEncoder(Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation='relu'),
            Dense(embed_dim),
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training=None):
        attn_output = self.att(inputs, inputs, training=training)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)
    
# === Positional Embedding Layer ===
class TimePositionalEmbedding(Layer):
    def __init__(self, sequence_length, embed_dim):
        super().__init__()
        self.pos_embedding = Embedding(input_dim=sequence_length, output_dim=embed_dim)

    def call(self, x):
        seq_len = tf.shape(x)[1]
        positions = tf.range(start=0, limit=seq_len, delta=1)
        pos_encoding = self.pos_embedding(positions)
        return x + pos_encoding

# === Expanded search space including positional embedding toggle ===
search_space = {
    'embed_dim': [8, 16, 32, 64],
    'num_heads': [1, 2, 4, 8],
    'dropout_rate': [0.1, 0.2, 0.3],
    'learning_rate': [1e-5, 3e-5, 1e-4, 3e-4, 1e-3],
    'use_positional_embedding': [True, False]
}

# === Random sampler with filtering ===
def random_sample_combinations(space, num_samples=30):
    valid_combinations = []
    attempts = 0
    max_attempts = 500

    while len(valid_combinations) < num_samples and attempts < max_attempts:
        embed_dim = random.choice(space['embed_dim'])
        num_heads = random.choice(space['num_heads'])
        dropout_rate = random.choice(space['dropout_rate'])
        learning_rate = random.choice(space['learning_rate'])
        use_positional_embedding = random.choice(space['use_positional_embedding'])

        if embed_dim % num_heads != 0:
            attempts += 1
            continue

        ff_dim = random.choice([embed_dim, 2 * embed_dim, 4 * embed_dim])

        config = {
            'embed_dim': embed_dim,
            'num_heads': num_heads,
            'ff_dim': ff_dim,
            'dropout_rate': dropout_rate,
            'learning_rate': learning_rate,
            'use_positional_embedding': use_positional_embedding,
        }
        valid_combinations.append(config)
        attempts += 1

    if len(valid_combinations) < num_samples:
        print(f"Warning: Only generated {len(valid_combinations)} valid configs out of requested {num_samples}")

    return valid_combinations

# == Positional Embedding Layer Definition ==
class TimePositionalEmbedding(Layer):
    def __init__(self, sequence_length, embed_dim):
        super().__init__()
        self.pos_embedding = Embedding(input_dim=sequence_length, output_dim=embed_dim)

    def call(self, x):
        seq_len = tf.shape(x)[1]
        positions = tf.range(start=0, limit=seq_len, delta=1)
        pos_encoding = self.pos_embedding(positions)
        return x + pos_encoding
    
# === Updated model builder with positional embedding toggle ===
def build_transformer_model(input_shape, embed_dim=8, num_heads=2, ff_dim=16, dropout_rate=0.2, use_positional_embedding=True):
    inputs = Input(shape=input_shape)
    x = Dense(embed_dim)(inputs)
    if use_positional_embedding:
        x = TimePositionalEmbedding(sequence_length=input_shape[0], embed_dim=embed_dim)(x)
    x = TransformerEncoder(embed_dim=embed_dim, num_heads=num_heads, ff_dim=ff_dim)(x)
    x = GlobalAveragePooling1D()(x)
    x = Dense(32, activation='relu')(x)
    x = Dropout(dropout_rate)(x)
    outputs = Dense(1)(x)
    return Model(inputs=inputs, outputs=outputs)

# === Full search routine with TensorBoard and printouts ===
def run_transformer_search(input_shape, train_ds, val_ds, trials=30, log_dir_root='logs/hparam_search'):
    best_mae = float('inf')
    best_config = None
    results = []

    configs = random_sample_combinations(search_space, trials)

    for i, config in enumerate(configs):
        print(f"\n🧪 Trial {i+1}/{len(configs)} with config:\n{config}")
        K.clear_session()

        model = build_transformer_model(
            input_shape=input_shape,
            embed_dim=config['embed_dim'],
            num_heads=config['num_heads'],
            ff_dim=config['ff_dim'],
            dropout_rate=config['dropout_rate'],
            use_positional_embedding=config['use_positional_embedding']
        )

        optimizer = tf.keras.optimizers.Adam(learning_rate=config['learning_rate'])
        model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

        timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
        log_dir = os.path.join(log_dir_root, f"trial_{i+1}_{timestamp}")
        os.makedirs(log_dir, exist_ok=True)

        callbacks = [
            EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True),
            TensorBoard(log_dir=log_dir, histogram_freq=1)
        ]

        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=70,
            callbacks=callbacks,
            verbose=0
        )

        val_mae = min(history.history['val_mae'])
        results.append((config, val_mae))
        all_maes.append(val_mae)

        # Determine best/worst seen *so far* for emoji scoring
        current_best_mae = min(all_maes)
        current_worst_mae = max(all_maes)
        emoji = pick_emoji(val_mae, current_best_mae, current_worst_mae)
        print(f"\n{emoji} Trial {i+1} finished — Validation MAE: {val_mae:.4f}")

        # Update best config if *this* trial is best
        if val_mae < best_mae:
            best_mae = val_mae
            best_config = config


    print("\n🏆 Best Configuration Found:")
    print(best_config)
    print(f"🎯 Best Validation MAE: {best_mae:.4f}")

    return best_config, results


In [ ]:
best_config, all_results = run_transformer_search(input_shape, train_ds, val_ds, trials=20)


🧪 Trial 1/20 with config:
{'embed_dim': 8, 'num_heads': 2, 'ff_dim': 8, 'dropout_rate': 0.2, 'learning_rate': 3e-05, 'use_positional_embedding': False}

👌 Trial 1 finished — Validation MAE: 0.3885

🧪 Trial 2/20 with config:
{'embed_dim': 8, 'num_heads': 4, 'ff_dim': 8, 'dropout_rate': 0.2, 'learning_rate': 3e-05, 'use_positional_embedding': True}

🌟 Trial 2 finished — Validation MAE: 0.3612

🧪 Trial 3/20 with config:
{'embed_dim': 64, 'num_heads': 1, 'ff_dim': 64, 'dropout_rate': 0.2, 'learning_rate': 0.0001, 'use_positional_embedding': True}

🚀 Trial 3 finished — Validation MAE: 0.1829

🧪 Trial 4/20 with config:
{'embed_dim': 64, 'num_heads': 2, 'ff_dim': 64, 'dropout_rate': 0.1, 'learning_rate': 0.0003, 'use_positional_embedding': False}

🌟 Trial 4 finished — Validation MAE: 0.1930

🧪 Trial 5/20 with config:
{'embed_dim': 32, 'num_heads': 1, 'ff_dim': 128, 'dropout_rate': 0.1, 'learning_rate': 1e-05, 'use_positional_embedding': False}

🛠️ Trial 5 finished — Validation MAE: 0.2952

🧪

In [ ]:
best_config = {'embed_dim': 32, 'num_heads': 2, 'ff_dim': 64, 'dropout_rate': 0.1, 'learning_rate': 0.001, 'use_positional_embedding': True}

## Test Best Transformer Parameters

In [ ]:
input_shape = (X.shape[1], X.shape[2])  # timesteps, features

keras.backend.clear_session()
optimizer = tf.keras.optimizers.Adam(learning_rate=best_config['learning_rate'])
model = build_transformer_model(
            input_shape=input_shape,
            embed_dim=best_config['embed_dim'],
            num_heads=best_config['num_heads'],
            ff_dim=best_config['ff_dim'],
            dropout_rate=best_config['dropout_rate'],
            use_positional_embedding=best_config['use_positional_embedding']
        )
model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(monitor='val_loss', restore_best_weights=True, patience=30)

history = model.fit(train_ds,
                    epochs=100,
                    validation_data=val_ds,
                    callbacks=[early_stop])


### Create Plotting Functions

In [ ]:
plot_training_history(history, log_scale=True)

### Test model on full dataset

In [ ]:
# Load new data — it should look like your original df
new_file_df = data[data['file'] == 'datasets/256_Al/Dv1.0']  # or whatever new file

# Make sure it's sorted by timestep
new_file_df = new_file_df.sort_values('timestep')
new_file_df

In [ ]:
pred_sequences = []

for i in range(len(new_file_df) - timesteps):
    window = new_file_df.iloc[i:i+timesteps][X_cols].values
    pred_sequences.append(window)

X_pred = np.array(pred_sequences)  # shape: (num_windows, timesteps, num_features)


In [ ]:
y_pred = model.predict(X_pred)  # shape: (num_windows, 1)
y_pred = y_pred.flatten()


In [ ]:
from sklearn.metrics import mean_absolute_error

def plot_predictions_with_zoom(y_true, y_pred, y_col='target', offset=0,
                                zoom_range=(3000, 3200), log_scale=True):
    """
    Plots predicted vs. true values with MAE and a zoomed-in subplot.

    Parameters:
    - y_true: Array of ground truth values (will be shifted by offset).
    - y_pred: Array of predicted values.
    - y_col: Name of the target variable (for axis labeling).
    - offset: Integer offset if prediction starts after timesteps.
    - zoom_range: Tuple of (x_min, x_max) for the zoomed-in view.
    - log_scale: Whether to apply symlog scale to the y-axis.
    """
    if offset:
        y_true = y_true[offset:]
    
    mae = mean_absolute_error(y_true, y_pred)

    fig, axs = plt.subplots(2, 1, figsize=(12, 7), sharey=True)

    # --- Full Plot ---
    axs[0].plot(y_true, label=f'True {y_col}', linewidth=2)
    axs[0].plot(y_pred, label=f'Predicted {y_col}', linewidth=2, linestyle='--')
    axs[0].set_title(f'Predicted vs. True {y_col.capitalize()} (Full)\nMAE = {mae:.4f} $E_h$', fontsize=14)
    axs[0].set_ylabel(f'{y_col.capitalize()}')
    axs[0].legend()
    axs[0].grid(True, linestyle='--', linewidth=0.5)
    if log_scale:
        axs[0].set_yscale("symlog")

    # --- Zoomed-In Plot ---
    axs[1].plot(y_true, label=f'True {y_col}', linewidth=2)
    axs[1].plot(y_pred, label=f'Predicted {y_col}', linewidth=2, linestyle='--')
    axs[1].set_title(f'Zoomed In: Steps {zoom_range[0]}–{zoom_range[1]}', fontsize=13)
    axs[1].set_xlabel('Time Step')
    axs[1].set_ylabel(f'{y_col.capitalize()}')
    axs[1].set_xlim(zoom_range)
    axs[1].legend()
    axs[1].grid(True, linestyle='--', linewidth=0.5)
    if log_scale:
        axs[1].set_yscale("symlog")

    plt.tight_layout()
    plt.show()



In [ ]:
plot_predictions_with_zoom(
    y_true=new_file_df[y_col].values,
    y_pred=y_pred,
    y_col=y_col,
    offset=timesteps,
    zoom_range=(3000, 3200),
    log_scale=False
)


## Tune Parameters for Recurrent Neural Network

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Activation, Bidirectional, Normalization
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K
import numpy as np
import random

# === Example Input ===
# X: shape = (samples, timesteps, features)
# train_ds, val_ds: tf.data.Dataset objects
# This example assumes X is already defined.
# If not, define X, train_ds, and val_ds accordingly.

# TO DO
# Redo kernel and recurrent later with batchsize and timestep later

# === Search Space ===
search_space = {
    'lstm_units': [4, 8, 16, 32, 64],
    'dropout_rate': [0.1, 0.2],
    'dense_units': [0, 4, 8, 16, 32],
    'learning_rate': [1e-4, 3e-4, 1e-3],
    'l1_kernel': [0.0, 0.001, 0.005],
    'l1_recurrent': [0.0, 0.001, 0.005],
    'activation': ['relu']
}

# === Random Config Sampler ===
def random_sample_configs(space, num_trials=10):
    configs = []
    for _ in range(num_trials):
        config = {
            'lstm_units': random.choice(space['lstm_units']),
            'dropout_rate': random.choice(space['dropout_rate']),
            'dense_units': random.choice(space['dense_units']),
            'learning_rate': random.choice(space['learning_rate']),
            'l1_kernel': random.choice(space['l1_kernel']),
            'l1_recurrent': random.choice(space['l1_recurrent']),
            'activation': random.choice(space['activation'])
        }
        configs.append(config)
    return configs

# === Model Builder ===
def build_bidirectional_lstm_model(input_shape, config, norm_layer):
    kernel_reg = regularizers.L1(config['l1_kernel']) if config['l1_kernel'] > 0 else None
    recurrent_reg = regularizers.L1(config['l1_recurrent']) if config['l1_recurrent'] > 0 else None

    model = Sequential()
    model.add(norm_layer)

    model.add(Bidirectional(
        LSTM(config['lstm_units'], # Change to constant 8
             return_sequences=False,
             kernel_regularizer=kernel_reg,
             recurrent_regularizer=recurrent_reg),
        input_shape=input_shape
    ))
    if config['dense_units'] != 0:
        model.add(Dropout(0.2))
        model.add(Dense(config['dense_units'])) # Change to 8 constant
        model.add(Activation('relu'))
    model.add(Dropout(0.2))
    model.add(Dense(1))  # Output: regression scalar

    return model

# === Hyperparameter Search Runner ===
def run_bidirectional_lstm_search(X, train_ds, val_ds, num_trials=60):
    input_shape = (X.shape[1], X.shape[2])

    # Normalize
    norm_layer = Normalization()
    X_flat = X.reshape(-1, X.shape[-1])
    norm_layer.adapt(X_flat)

    best_mae = float('inf')
    best_config = None
    results = []

    configs = random_sample_configs(search_space, num_trials)

    for i, config in enumerate(configs):
        print(f"\n🔁 Trial {i+1}/{num_trials} | Config: {config}")
        K.clear_session()

        model = build_bidirectional_lstm_model(input_shape, config, norm_layer)
        optimizer = tf.keras.optimizers.Adam(learning_rate=config['learning_rate'])
        model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

        early_stop = EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True)

        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=70,
            callbacks=[early_stop],
            verbose=0
        )

        val_mae = min(history.history['val_mae'])
        print(f"✅ Trial {i+1} finished — Best Validation MAE: {val_mae:.4f}")
        results.append((config, val_mae))

        if val_mae < best_mae:
            best_mae = val_mae
            best_config = config

    print("\n🏆 Best Configuration Found:")
    print(best_config)
    print(f"🎯 Best Validation MAE: {best_mae:.4f}")

    return best_config, results


In [ ]:
best_config, results = run_bidirectional_lstm_search(X, train_ds, val_ds, num_trials=40)

## Test Best Recurrent Neural Network parameters

In [ ]:
best_config = {'lstm_units': 8, 'dropout_rate': 0.2, 'dense_units': 16, 'learning_rate': 0.0003, 'l1_kernel': 0.0, 'l1_recurrent': 0.001, 'activation': 'relu'}

In [ ]:
# # Assume X shape is (samples, timesteps, features)
# X_flat = X.reshape(-1, X.shape[-1])  # collapse time dimension
# X_pred

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, SimpleRNN, Normalization, BatchNormalization, Activation

# norm_layer = Normalization()
# norm_layer.adapt(X_flat)  # learns mean/std for each feature across time and samples

# model = Sequential([
#     norm_layer,
#     LSTM(64, input_shape=(X.shape[1], X.shape[2]), return_sequences=True),
#     Dropout(0.2),
#     LSTM(64, input_shape=(X.shape[1], X.shape[2]), return_sequences=False),
#     Dense(32), #, activation='relu'), # removed relu since target can be negative
#     Dropout(0.2),
#     Dense(1)
# ])

# model = Sequential([
#     norm_layer,
#     LSTM(64, input_shape=(X.shape[1], X.shape[2])),
#     Dropout(0.2),
#     Dense(32, activation='relu'),
#     Dense(1)
# ])
#{'lstm_units': 16, 'dropout_rate': 0.1, 'dense_units': 32, 'learning_rate': 0.001, 'l1_kernel': 0.005, 'l1_recurrent': 0.0, 'activation': 'relu'}

import tensorflow as tf
from tensorflow.keras import regularizers
from tensorflow.keras.layers import Bidirectional, LSTM, Dropout, Dense, Activation, Normalization
from tensorflow.keras.models import Sequential
from tensorflow.keras import backend as K
from tensorflow.keras import Input

# Clear previous TF session
K.clear_session()

# def build_bidirectional_lstm_model(input_shape, config, norm_layer):
#     kernel_reg = regularizers.L1(config['l1_kernel']) if config['l1_kernel'] > 0 else None
#     recurrent_reg = regularizers.L1(config['l1_recurrent']) if config['l1_recurrent'] > 0 else None

#     model = Sequential([
#         norm_layer,
#         Bidirectional(
#             LSTM(config['lstm_units'],
#                  return_sequences=False,
#                  kernel_regularizer=kernel_reg,
#                  recurrent_regularizer=recurrent_reg),
#             input_shape=input_shape
#         ),
#         Dropout(config['dropout_rate']),
#         Dense(config['dense_units']),
#         Activation(config['activation']),
#         Dropout(config['dropout_rate']),
#         Dense(1)
#     ])

#     return model

def build_bidirectional_lstm_model(input_shape, config, norm_layer):
    kernel_reg = regularizers.L1(config['l1_kernel']) if config['l1_kernel'] > 0 else None
    recurrent_reg = regularizers.L1(config['l1_recurrent']) if config['l1_recurrent'] > 0 else None

    model = Sequential()
    model.add(Input(shape=input_shape))
    model.add(norm_layer)

    model.add(Bidirectional(
        LSTM(config['lstm_units'],
             return_sequences=False,
             kernel_regularizer=kernel_reg,
             recurrent_regularizer=recurrent_reg)
    ))
    model.add(Dropout(0.1))
    model.add(Dense(16))
    model.add(Dropout(0.1))
    model.add(Dense(1))  # Regression output

    return model

# Example usage:
# Assume X is your input numpy array with shape (samples, timesteps, features)
X_flat = X.reshape(-1, X.shape[-1])
norm_layer = Normalization()
norm_layer.adapt(X_flat)

input_shape = (X.shape[1], X.shape[2])

In [ ]:
model = build_bidirectional_lstm_model(input_shape=input_shape, config=best_config, norm_layer=norm_layer)

optimizer = tf.keras.optimizers.Adam(learning_rate=best_config['learning_rate'])
model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=50, restore_best_weights=True)

history = model.fit(train_ds,
                    epochs=200,
                    validation_data=val_ds,
                    callbacks=[early_stop])

In [ ]:
plot_loglog_mae(history)
plot_training_history(history, log_scale=False)

## Plot RNN predictions

In [ ]:
# Load new data — it should look like your original df
new_file_df = data[data['file'] == 'datasets/256_Al_channel/Dv1.0']  # or whatever new file

# Make sure it's sorted by timestep
new_file_df = new_file_df.sort_values('timestep')


In [ ]:
pred_sequences = []

for i in range(len(new_file_df) - timesteps):
    window = new_file_df.iloc[i:i+timesteps][X_cols].values
    pred_sequences.append(window)

X_pred = np.array(pred_sequences)  # shape: (num_windows, timesteps, num_features)
X_pred

In [ ]:
y_pred = model.predict(X_pred)  # shape: (num_windows, 1)
y_pred = y_pred.flatten()

In [ ]:
plot_predictions_with_zoom(
    y_true=new_file_df[y_col].values,
    y_pred=y_pred,
    y_col=y_col,
    offset=timesteps,
    zoom_range=(3000, 3200)
)

### Old style graphs

In [ ]:
from sklearn.metrics import mean_absolute_error

y_true = new_file_df[y_col].values[timesteps:]  # shift due to prediction offset

mae = mean_absolute_error(y_true, y_pred)

plt.figure(figsize=(10, 4))
plt.plot(y_true, label=f'True {y_col}', linewidth=2)
plt.plot(y_pred, label=f'Predicted {y_col}', linewidth=2, linestyle='--')
plt.xlabel('Time Step')
plt.ylabel(f'{y_col.capitalize()}')
plt.title(f'Predicted vs. True {y_col}\nMAE = {mae:.4f} $E_h$')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import mean_absolute_error

y_true = new_file_df[y_col].values[timesteps:]  # shift due to prediction offset

mae = mean_absolute_error(y_true, y_pred)

plt.figure(figsize=(10, 4))
plt.plot(y_true, label=f'True {y_col}', linewidth=2)
plt.plot(y_pred, label=f'Predicted {y_col}', linewidth=2, linestyle='--')
plt.xlabel('Time Step')
plt.ylabel(f'{y_col.capitalize()}')
plt.title(f'Predicted vs. True Energy\nMAE = {mae:.4f} $E_h$')
plt.legend()
plt.xlim(3000,3200)
plt.grid(True)
plt.tight_layout()
plt.show()